# Week 3 Assignment
## SQL Subqueries, CTEs and Window Functions

**Name:** Soham Deshmukh

**Dataset:** Sample - Superstore.csv

**Objective:** Use Subqueries, CTEs, and Window Functions to analyze sales data from the Superstore dataset. 

**Step 1**:  DATA SETUP 

In [2]:
#Import required library
import pandas as pd
import sqlite3

#Load the Superstore dataset in dataframe
df = pd.read_csv("../dataset/Sample - Superstore.csv", encoding="latin1")

#Display first 5 rows
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [3]:
#Check dataset information
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9994 non-null   int64  
 1   Order ID       9994 non-null   str    
 2   Order Date     9994 non-null   str    
 3   Ship Date      9994 non-null   str    
 4   Ship Mode      9994 non-null   str    
 5   Customer ID    9994 non-null   str    
 6   Customer Name  9994 non-null   str    
 7   Segment        9994 non-null   str    
 8   Country        9994 non-null   str    
 9   City           9994 non-null   str    
 10  State          9994 non-null   str    
 11  Postal Code    9994 non-null   int64  
 12  Region         9994 non-null   str    
 13  Product ID     9994 non-null   str    
 14  Category       9994 non-null   str    
 15  Sub-Category   9994 non-null   str    
 16  Product Name   9994 non-null   str    
 17  Sales          9994 non-null   float64
 18  Quantity       9994

In [4]:
#Create a sqlite database connection
conn = sqlite3.connect("superstore.db")
cursor = conn.cursor()

In [5]:
#Store the dataset into a SQL table
df.to_sql("superstore_raw", conn, if_exists="replace", index=False)

print("Dataset imported successfully.")

Dataset imported successfully.


In [6]:
#creating required tables

#create customers table from the raw dataset

cursor.execute("""
CREATE TABLE customers AS
SELECT DISTINCT
               "Customer ID",
               "Customer Name",
               Segment,
               Country,
               City,
               State,
               Region
FROM superstore_raw;
""")

conn.commit()

print("Customers table created.")

Customers table created.


In [7]:
#create products table

cursor.execute("""
CREATE TABLE products AS
SELECT DISTINCT
               "Product ID",
               Category,
               "Sub-Category",
               "Product Name"
FROM superstore_raw;
""")

conn.commit()

print("Products table created.")

Products table created.


In [8]:
#create orders table

cursor.execute("""
CREATE TABLE orders AS
SELECT DISTINCT
               "Order ID",
               "Order Date",
               "Ship Date",
               "Ship Mode",
               "Customer ID",
               "Product ID",
               Sales,
               Quantity,
               Discount,
               Profit
FROM superstore_raw;
""")

conn.commit()

print("Orders table created.")

Orders table created.


**Step 2**: Perform Required Queries

**Question 1**

Find all orders where sales are greater than the average sales.

In [ ]:
#the average sales 
query =""" SELECT AVG(Sales) FROM orders;"""
pd.read_sql(query,conn)

,AVG(Sales)
0,229.852846


In [9]:
query = """
SELECT * FROM orders
WHERE Sales > (SELECT AVG(Sales) FROM orders);
"""

pd.read_sql(query, conn)

,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Product ID,Sales,Quantity,Discount,Profit
0,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,FUR-BO-10001798,261.9600,2,0.00,41.9136
1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,FUR-CH-10000454,731.9400,3,0.00,219.5820
2,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,FUR-TA-10000577,957.5775,5,0.45,-383.0310
3,CA-2014-115812,6/9/2014,6/14/2014,Standard Class,BH-11710,TEC-PH-10002275,907.1520,6,0.20,90.7152
4,CA-2014-115812,6/9/2014,6/14/2014,Standard Class,BH-11710,FUR-TA-10001539,1706.1840,9,0.20,85.3092
...,...,...,...,...,...,...,...,...,...,...
2354,US-2016-103674,12/6/2016,12/10/2016,Standard Class,AP-10720,TEC-PH-10004080,271.9600,5,0.20,27.1960
2355,US-2016-103674,12/6/2016,12/10/2016,Standard Class,AP-10720,TEC-PH-10002496,249.5840,2,0.20,31.1980
2356,US-2016-103674,12/6/2016,12/10/2016,Standard Class,AP-10720,OFF-BI-10002026,437.4720,14,0.20,153.1152
2357,CA-2017-121258,2/26/2017,3/3/2017,Standard Class,DB-13060,TEC-PH-10003645,258.5760,2,0.20,19.3932


**Question 2**

Find the highest sales order for each customer.

In [10]:
query = """
SELECT * FROM orders o
WHERE Sales = (SELECT MAX(Sales) FROM orders WHERE "Customer ID" = o."Customer ID");
"""

pd.read_sql(query, conn)

,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Product ID,Sales,Quantity,Discount,Profit
0,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,FUR-CH-10000454,731.9400,3,0.00,219.5820
1,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,FUR-TA-10000577,957.5775,5,0.45,-383.0310
2,CA-2014-115812,6/9/2014,6/14/2014,Standard Class,BH-11710,FUR-TA-10001539,1706.1840,9,0.20,85.3092
3,CA-2015-106320,9/25/2015,9/30/2015,Standard Class,EB-13870,FUR-TA-10000577,1044.6300,3,0.00,240.2649
4,US-2015-150630,9/17/2015,9/21/2015,Standard Class,TB-21520,FUR-BO-10004834,3083.4300,7,0.50,-1665.0522
...,...,...,...,...,...,...,...,...,...,...
790,CA-2015-159534,3/20/2015,3/23/2015,First Class,DH-13075,OFF-BI-10003656,1087.9360,8,0.20,353.5792
791,CA-2016-129630,9/4/2016,9/4/2016,Same Day,IM-15055,TEC-CO-10003763,2799.9600,5,0.20,944.9865
792,CA-2017-121559,6/1/2017,6/3/2017,Second Class,HW-14935,OFF-AP-10002945,2405.2000,8,0.00,793.7160
793,CA-2017-153871,12/11/2017,12/17/2017,Standard Class,RB-19435,OFF-BI-10004600,735.9800,2,0.00,331.1910


**Question 3**

Calculate total sales for each customer using a CTE.

In [11]:
query = """
WITH customer_sales AS (SELECT "Customer ID", SUM(Sales) AS total_sales FROM orders GROUP BY "Customer ID")
SELECT * FROM customer_sales;
"""

pd.read_sql(query, conn)

,Customer ID,total_sales
0,AA-10315,5563.560
1,AA-10375,1056.390
2,AA-10480,1790.512
3,AA-10645,5086.935
4,AB-10015,886.156
...,...,...
788,XP-21865,2374.658
789,YC-21895,5454.350
790,YS-21880,6720.444
791,ZC-21910,8025.707


**Question 4**

Find customers whose total sales are above average.

In [12]:
query = """
WITH customer_sales AS (SELECT "Customer ID", SUM(Sales) AS total_sales FROM orders GROUP BY "Customer ID")
SELECT * FROM customer_sales
WHERE total_sales > (SELECT AVG(total_sales) FROM customer_sales);
"""

pd.read_sql(query, conn)

,Customer ID,total_sales
0,AA-10315,5563.560
1,AA-10645,5086.935
2,AB-10060,7755.620
3,AB-10105,14473.571
4,AC-10450,5527.846
...,...,...
289,VW-21775,6134.038
290,WB-21850,6160.102
291,YC-21895,5454.350
292,YS-21880,6720.444


**Question 5**

Rank all customers based on total sales.

In [13]:
query = """
WITH customer_sales AS (SELECT "Customer ID", SUM(Sales) AS total_sales FROM orders GROUP BY "Customer ID")
SELECT "Customer ID", total_sales, RANK() OVER(ORDER BY total_sales DESC) AS customer_rank
FROM customer_sales;
"""

pd.read_sql(query, conn)

,Customer ID,total_sales,customer_rank
0,SM-20320,25043.050,1
1,TC-20980,19052.218,2
2,RB-19360,15117.339,3
3,TA-21385,14595.620,4
4,AB-10105,14473.571,5
...,...,...,...
788,RS-19870,22.328,789
789,MG-18205,16.739,790
790,CJ-11875,16.520,791
791,LD-16855,5.304,792


**Question 6**

Assign row numbers to each order within a customer.

In [14]:
query = """
SELECT "Order ID", "Customer ID", Sales, ROW_NUMBER() OVER(PARTITION BY "Customer ID" ORDER BY Sales DESC) AS row_number
FROM orders;
"""

pd.read_sql(query, conn)

,Order ID,Customer ID,Sales,row_number
0,CA-2016-103982,AA-10315,3930.072,1
1,CA-2014-128055,AA-10315,673.568,2
2,CA-2016-103982,AA-10315,431.976,3
3,CA-2017-147039,AA-10315,362.940,4
4,CA-2014-128055,AA-10315,52.980,5
...,...,...,...,...
9988,CA-2017-141481,ZD-21925,61.440,5
9989,CA-2014-143336,ZD-21925,22.720,6
9990,US-2016-147991,ZD-21925,16.720,7
9991,CA-2016-152471,ZD-21925,15.984,8


**Question 7**

Display the top 3 customers based on total sales.

In [15]:
query = """
WITH customer_sales AS (SELECT "Customer ID", SUM(Sales) AS total_sales FROM orders GROUP BY "Customer ID")
SELECT * FROM(SELECT "Customer ID", total_sales, RANK() OVER(ORDER BY total_sales DESC) AS customer_rank FROM customer_sales)
WHERE customer_rank <= 3;
"""

pd.read_sql(query, conn)

,Customer ID,total_sales,customer_rank
0,SM-20320,25043.050,1
1,TC-20980,19052.218,2
2,RB-19360,15117.339,3


**Step 3: Final Combined Query**

Display Customer Name, Total Sales and Rank using JOIN, CTE and Window Function.

In [ ]:
#checking for duplicates
query = """
SELECT "Customer ID", COUNT(*) AS total_rows
FROM customers
GROUP BY "Customer ID"
HAVING COUNT(*) > 1;
"""
#hence duplicate records in customer table identified
pd.read_sql(query, conn)

,Customer ID,total_rows
0,AA-10315,4
1,AA-10375,9
2,AA-10480,4
3,AA-10645,6
4,AB-10015,3
...,...,...
775,XP-21865,10
776,YC-21895,5
777,YS-21880,8
778,ZC-21910,12


In [24]:
#displaying customer name, total sales and rank together
query = """
WITH customer_sales AS(SELECT "Customer ID", SUM(Sales) AS total_sales FROM orders GROUP BY "Customer ID")
SELECT DISTINCT c."Customer Name", cs.total_sales, RANK() OVER(ORDER BY cs.total_sales DESC) AS customer_rank
FROM customer_sales cs
JOIN customers c
ON cs."Customer ID" = c."Customer ID";
"""

pd.read_sql(query, conn)

,Customer Name,total_sales,customer_rank
0,Sean Miller,25043.050,1
1,Tamara Chand,19052.218,6
2,Raymond Buch,15117.339,11
3,Tom Ashbrook,14595.620,17
4,Adrian Barton,14473.571,20
...,...,...,...
788,Roy Skaria,22.328,4682
789,Mitch Gastineau,16.739,4684
790,Carl Jackson,16.520,4685
791,Lela Donovan,5.304,4686


**Note:** in above query customer rank is skipped due to multiple customer having similar total sales

## Mini Project: Customer Sales Insights

In [17]:
# displaying the top five customers based on total sales as follows
query = """
WITH customer_sales AS(SELECT "Customer ID", SUM(Sales) AS total_sales FROM orders GROUP BY "Customer ID")
SELECT c."Customer Name", cs.total_sales 
FROM customer_sales cs
JOIN customers c
ON cs."Customer ID" = c."Customer ID"
ORDER BY total_sales DESC
LIMIT 5;
"""

pd.read_sql(query, conn)

,Customer Name,total_sales
0,Sean Miller,25043.05
1,Sean Miller,25043.05
2,Sean Miller,25043.05
3,Sean Miller,25043.05
4,Sean Miller,25043.05


In [18]:
#displaying the bottom five customers based on total sales as follows
query = """
WITH customer_sales AS(SELECT "Customer ID", SUM(Sales) AS total_sales FROM orders GROUP BY "Customer ID")
SELECT c."Customer Name", cs.total_sales 
FROM customer_sales cs
JOIN customers c
ON cs."Customer ID" = c."Customer ID"
ORDER BY total_sales ASC
LIMIT 5;
"""

pd.read_sql(query, conn)

,Customer Name,total_sales
0,Thais Sissman,4.833
1,Thais Sissman,4.833
2,Lela Donovan,5.304
3,Carl Jackson,16.520
4,Mitch Gastineau,16.739


In [19]:
#displaying customers who placed only one order as follows
query = """
SELECT c."Customer Name", COUNT(o."Order ID") AS total_orders
FROM customers c
JOIN orders o
ON c."Customer ID" = o."Customer ID"
GROUP BY c."Customer Name"
HAVING COUNT(o."Order ID")= 1;
"""

pd.read_sql(query, conn)

,Customer Name,total_orders
0,Anthony O'Donnell,1
1,Carl Jackson,1
2,Jocasta Rupert,1
3,Lela Donovan,1
4,Ricardo Emerson,1


In [20]:
#displaying customers having above-average total sales as follows
query = """
WITH customer_sales AS(SELECT "Customer ID", SUM(Sales) AS total_sales FROM orders GROUP BY "Customer ID")
SELECT c."Customer Name", cs.total_sales
FROM customer_sales cs
JOIN customers c
ON cs."Customer ID" = c."Customer ID"
WHERE total_sales >(SELECT AVG(total_sales) FROM customer_sales);
"""

pd.read_sql(query, conn)

,Customer Name,total_sales
0,Brosina Hoffman,6255.351
1,Irene Maddox,4930.474
2,Pete Kriz,8646.934
3,Tracy Blumstein,4737.486
4,Matt Abelman,4299.161
...,...,...
2106,Maris LaWare,2921.500
2107,Ruben Ausman,3832.314
2108,Tom Boeckenhauer,9133.990
2109,Dave Brooks,4531.646


In [21]:
#displaying the highest order value for every customer as follows
query = """
SELECT c."Customer Name", MAX(o.Sales) AS highest_order_value 
FROM customers c
JOIN orders o
ON c."Customer ID" = o."Customer ID"
GROUP BY c."Customer Name"
ORDER BY highest_order_value DESC;
"""

pd.read_sql(query, conn)

,Customer Name,highest_order_value
0,Sean Miller,22638.480
1,Tamara Chand,17499.950
2,Raymond Buch,13999.960
3,Tom Ashbrook,11199.968
4,Hunter Lopez,10499.970
...,...,...
788,Carl Jackson,16.520
789,Mitch Gastineau,12.320
790,Roy Skaria,9.648
791,Lela Donovan,5.304


## What I Learned

This assignment helped me understand how SQL can be used to organize and analyze real-world sales data. I also learned how Subqueries, CTEs, and Window Functions like `RANK()` and `ROW_NUMBER()` simplify complex queries and make data analysis easier. Also I analyzed customer sales data to identify top customers, bottom customers, and customers with above-average sales.

